In [2]:
# Install Unsloth and its dependencies
!pip install unsloth
!pip install --no-deps trl accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━

In [ ]:
from huggingface_hub import login

# Replace 'your_token_here' with your actual Hugging Face access token
login(token='')

In [3]:
import torch, math, sqlite3, re, unicodedata
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "kazol196295/llama-3.1-8b-bengali-mental-health-counsellor",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)

# Save references so later cells don't overwrite them
finetuned_model     = model
finetuned_tokenizer = tokenizer

print("✅ Model loaded successfully")


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded successfully


In [5]:

from transformers import AutoTokenizer as _AutoTok
count_tokenizer = _AutoTok.from_pretrained("unsloth/Llama-3.1-8B-Instruct")

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [8]:
import re
import pandas as pd
from datasets import Dataset, DatasetDict

# ── Constants ─────────────────────────────────────────────────────────────────
DATASET_PATH  = "/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv"
ANSWER_MARKER = "[/INST]\nCounselor:\n"   # shared split key — never change

# ── 1. Load & clean ───────────────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

Dataset shape: (38233, 4)

Columns: ['Topics', 'Question-Title', 'Questions', 'Answers']

First few rows:


,Topics,Question-Title,Questions,Answers
0,পারিবারিক দ্বন্দ্ব,মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি,আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...,"আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা ""ত্রি..."
1,"পদার্থের অপব্যবহার, আসক্তি",আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?,"আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ...",হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্...
2,পারিবারিক দ্বন্দ্ব,আমার পরিবারের কাছ থেকে গোপন রাখা,"আমার মনের মধ্যে গোপন আছে, এবং আমি জানি না তাদ...",মনে হচ্ছে গোপন রাখা এখন আপনার জন্য একটি সমস্য...
3,"আচরণগত পরিবর্তন, সামাজিক সম্পর্ক",অধিকারী হওয়ার অন্তর্নিহিত কারণ,আমি আমার সম্পর্কের ক্ষেত্রে অত্যন্ত অধিকারসূচক...,হ্যালো। এটা দুর্দান্ত যে আপনি উপলব্ধি করতে সক...
4,দুশ্চিন্তা,আমি কি ওষুধ ছাড়া উদ্বেগ নিয়ন্ত্রণ করতে পারি?,কয়েক বছর আগে আমার মাথায় আঘাত লেগেছিল এবং আমা...,আপনি বলেননি কি বা কত ওষুধ আপনি চেষ্টা করেছেন।...


In [9]:
df["q_tokens"] = df["Questions"].apply(lambda x: len(count_tokenizer.encode(str(x))))
df["a_tokens"] = df["Answers"].apply(lambda x: len(count_tokenizer.encode(str(x))))
df["total_tokens"] = df.apply(
    lambda row: len(count_tokenizer.encode(
        f"[INST]\n{row['Questions']}\n[/INST]\nCounselor:\n{row['Answers']}"
    )), axis=1
)
df = df[df["a_tokens"] > 35]
df = df[df["total_tokens"] < 2048]
def format_prompt(row):
    return f"""[INST]
Topic: {row['Topics']}
User: {row['Questions']}
Respond empathetically as a counselor.
[/INST]
Counselor:
{row['Answers']}"""

df["text"] = df.apply(format_prompt, axis=1)
print(f"Filtered dataset size: {len(df)}")

Filtered dataset size: 29651


In [10]:
# Must use same seed=42 as training to get identical test set
full_dataset = Dataset.from_pandas(df)
split1   = full_dataset.train_test_split(test_size=0.05, seed=42)
split2   = split1["test"].train_test_split(test_size=0.10, seed=42)
test_ds  = split2["train"]  # same test set as training notebook
print(f"Test set size: {len(test_ds)}")

Test set size: 1334


In [11]:
test_ds  = split2["train"].select(range(5))  # same test set as training notebook
print(f"Test set size: {len(test_ds)}")

Test set size: 5


In [15]:
import logging
import transformers

# This silences the specific logger causing the crash
transformers.utils.logging.set_verbosity_error() 


def run_nlp_metrics(model, tokenizer, dataset):
    results = []
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    smooth = SmoothingFunction().method1
    eval_set = dataset.select(range(min(10, len(dataset))))

    print("Generating responses for BLEU / ROUGE...")
    for row in tqdm(eval_set):
        # ✅ Prompt WITHOUT answer — same indentation as training
        prompt = f"""[INST]
                    Topic: {row['Topics']}
                    User: {row['Questions']}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    """
        reference = str(row['Answers']).strip()

        inputs  = tokenizer(prompt, return_tensors="pt",
                            truncation=True, max_length=1024).to("cuda")
        out_ids = model.generate(
            **inputs, max_new_tokens=512, use_cache=True,
            temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        # Strip prompt tokens, decode only new generated tokens
        gen_ids    = out_ids[0][inputs["input_ids"].shape[1]:]
        prediction = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        r_score = scorer.score(reference, prediction)
        b_score = sentence_bleu([reference.split()], prediction.split(),
                                smoothing_function=smooth)

        results.append({
            "topic":      row['Topics'],
            "input":      row['Questions'],
            "reference":  reference,
            "prediction": prediction,
            "bleu":       b_score,
            "rouge1":     r_score['rouge1'].fmeasure,
            "rougeL":     r_score['rougeL'].fmeasure,
            "timestamp":  datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    return pd.DataFrame(results)

metrics_df = run_nlp_metrics(finetuned_model, finetuned_tokenizer, test_ds)
print(f"\nAvg BLEU   : {metrics_df['bleu'].mean():.4f}")
print(f"Avg ROUGE-1: {metrics_df['rouge1'].mean():.4f}")
print(f"Avg ROUGE-L: {metrics_df['rougeL'].mean():.4f}")

Generating responses for BLEU / ROUGE...


100%|██████████| 5/5 [03:01<00:00, 36.38s/it]


Avg BLEU   : 0.0030
Avg ROUGE-1: 0.0000
Avg ROUGE-L: 0.0000


In [16]:
print("--- Human Evaluation Sample ---")
# Pick 5 random samples to review
sample_eval = metrics_df.sample(5)

for i, row in sample_eval.iterrows():
    print(f"\n[TOPIC]: {row['topic']}")
    print(f"[USER]: {row['input']}")
    print(f"[MODEL RESPONSE]: {row['prediction']}")
    print(f"[HUMAN REFERENCE]: {row['reference']}")
    print("-" * 30)
    # Manual Task: Rate the MODEL RESPONSE from 1-5 on empathy quality.

--- Human Evaluation Sample ---

[TOPIC]: রাগান্বিত
[USER]: আমি পড়াশুনা করি কিন্তু আমি স্মার্ট নই।
[MODEL RESPONSE]: আমি নিশ্চিত আপনি কঠোর পড়াশোনা করছেন, কারণ আপনি আমাকে এই ভাবে বলেছেন। তোমার জন্য ভালো হবে। তোমার জন্য ভালো হবে। আপনার কোন অধ্যয়ন কৌশল আছে? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব
[HUMAN REFERENCE]: যতক্ষণ না তারা স্মার্ট হতে না শেখে ততক্ষণ পর্যন্ত কেউ স্মার্ট হয় না। আমরা সবাই মস্তিষ্ক নিয়ে জন্মেছি কিন্তু বুদ্ধি নিয়ে নয় আমাদের বুদ্ধি অর্জন করা উচিত।
------------------------------

[TOPIC]: বিস্মিত
[USER]: হ্যাঁ আমি সত্যিই খুশি ছিল
[MODEL RESPONSE]: আমি খুশি যে আপনি একটি বিস্ময়কর জন্মদিন ছিল. আমি আশা করি আপনি আপনার বন্ধুর সাথে সময় উপভোগ করেছেন। আমি আশা করি আপনি আপনার বন্ধুদের সাথে সময় উপভোগ করেছেন। আপনার জন্মদিন আপনার জন্য কিভাবে ছি

In [17]:
def human_eval(metrics_df, num_samples=5):
    print("\n=== Human Evaluation — Rate empathy quality (1=poor, 5=excellent) ===\n")
    sample = metrics_df.sample(min(num_samples, len(metrics_df)), random_state=42)
    scores = []

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\nSample {i}/{num_samples}")
        print("─" * 60)
        print(f"[TOPIC]    : {row['topic']}")
        print(f"[USER]     : {row['input']}")
        print(f"[RESPONSE] : {row['prediction']}")
        print("─" * 60)
        while True:
            try:
                score = float(input("Empathy score [1.0 – 5.0]: "))
                if 1.0 <= score <= 5.0:
                    scores.append(score)
                    break
                else:
                    print("Enter a number between 1.0 and 5.0")
            except ValueError:
                print("Invalid input — try again")

    avg = sum(scores) / len(scores) if scores else None
    print(f"\n✅ Average human empathy score: {avg:.2f} / 5.0")
    return avg

avg_human_score = human_eval(metrics_df, num_samples=5)


=== Human Evaluation — Rate empathy quality (1=poor, 5=excellent) ===


Sample 1/5
────────────────────────────────────────────────────────────
[TOPIC]    : হতাশ
[USER]     : আপনি গত রাতে যে লটারির টিকিট কিনেছিলেন তা জেতেনি।
[RESPONSE] : আমি জানি এটা সত্যিই হতাশাজনক হতে পারে. আপনি কত টিকিট কিনেছেন? আপনি কখনই জেততে পারেন না। এই ভুল বিল আপনার টাকা নষ্ট করে দেয়। আপনি কি আরও টিকিট কিনতে যাচ্ছেন? আমি মনে করি আমি যদি আমার অর্থের সমস্ত পাঠ দিতাম তাহলে আমি জেতব। আমি মনে করি আমার একটি অবশিষ্ট কার্ড আছে। আমি আপনাকে শুভকামনা জানাচ্ছি! আপনি কি করেছেন যে আপনার জেতার পরিকল্পনা ভেঙে দেয়? আপনি কি আরও টিকিট কিনতে যাচ্ছেন? আপনি কি একটি বন্ধু আছে যে আপনি একটি টিকিট পেতে পারেন? আপনি কি একট
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  4



Sample 2/5
────────────────────────────────────────────────────────────
[TOPIC]    : বিস্মিত
[USER]     : হ্যাঁ আমি সত্যিই খুশি ছিল
[RESPONSE] : আমি খুশি যে আপনি একটি বিস্ময়কর জন্মদিন ছিল. আমি আশা করি আপনি আপনার বন্ধুর সাথে সময় উপভোগ করেছেন। আমি আশা করি আপনি আপনার বন্ধুদের সাথে সময় উপভোগ করেছেন। আপনার জন্মদিন আপনার জন্য কিভাবে ছিল? আমি আশা করি আপনি আপনার বন্ধুর সাথে সময় উপভোগ করেছেন। আপনি কি আপনার জন্মদিনের জন্য একটি পার্টি ছিল? আপনি কি আপনার বন্ধুর সাথে সময় উপভোগ করেছেন? আপনি কি আপনার জন্মদিনের জন্য একটি পার্টি ছিল? আপনি কি আপনার বন্ধুর সাথে সময় উপভোগ করেছেন
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  5



Sample 3/5
────────────────────────────────────────────────────────────
[TOPIC]    : একাকী
[USER]     : আমি গির্জা এবং বারে যাওয়ার চেষ্টা করেছি। আমি একা থাকতে খুব অপছন্দ করি.
[RESPONSE] : এটা কি সত্যি? আপনি কি এখনও নামাজ পড়েন? আমি আশা করি আপনি কিছু বন্ধু খুঁজে পাবেন। এটা সম্ভবত একটি ভাল ধারণা যেতে এবং একটি পার্টি আছে. আমি আশা করি আপনি একটি ভাল সময় আছে! আমি আপনার জন্য খুব উত্তেজিত! আমি আশা করি আপনি কিছু মজার মত পাবেন! আপনার বন্ধুদের সাথে কিছু আনন্দ করুন এবং আপনার বন্ধুদের সাথে কিছু মজা করুন! আমি আশা করি আপনি একটি ভাল সময় আছে! আমি আপনার জন্য খুব উত্তেজিত! আমি আশা করি আপনি কিছু মজার মত পাবেন! আপনার বন্ধুদে
────────────────────────────────────────────────────────────


...


Sample 4/5
────────────────────────────────────────────────────────────
[TOPIC]    : আশাবাদী
[USER]     : ভয় সাধারণত সব লাগে।
[RESPONSE] : হ্যাঁ, আমি আশা করি আপনি অনুভব করেন যে আপনার কাছে এটি সম্পর্কে কিছু করার আরও অনেক কিছু আছে। আমি বাজি ধরছি আপনি এটি সম্পর্কে খুব নার্ভাস। আপনি কি এটি সম্পর্কে কিছু করার কথা ভেবেছেন? হয়তো কিছু সহায়ক অনুষ্ঠান বা সাইকোলজি সেশন? এটি আপনাকে এমন একটি উপায় দেবে যা আপনি অনুভব করেন যে আপনি কাজ করছেন এবং আপনার এবং আপনার প্রিয়জনের সম্পর্কের উপর ভিত্তি করে আপনি আপনার জীবনে যেতে পারেন। আমি আশা করি আপনি সবকিছু ভাল করবেন. আপনি এই মুহূর্�
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  4



Sample 5/5
────────────────────────────────────────────────────────────
[TOPIC]    : রাগান্বিত
[USER]     : আমি পড়াশুনা করি কিন্তু আমি স্মার্ট নই।
[RESPONSE] : আমি নিশ্চিত আপনি কঠোর পড়াশোনা করছেন, কারণ আপনি আমাকে এই ভাবে বলেছেন। তোমার জন্য ভালো হবে। তোমার জন্য ভালো হবে। আপনার কোন অধ্যয়ন কৌশল আছে? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব্যবহার করেন? আপনি কি আপনার পড়াশোনার সাথে সময় ব
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  5



✅ Average human empathy score: 4.20 / 5.0


In [19]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime

def generate_response(model, tokenizer, topic, question, max_new_tokens=512):
    prompt = f"""[INST]
                    Topic: {topic}
                    User: {question}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    """
    inputs  = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=2048).to("cuda")
    out_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )
    gen_ids    = out_ids[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return prediction

# Generate for test set
print("Generating responses...")
records = []

for row in tqdm(test_ds):
    prediction = generate_response(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions']
    )
    records.append({
        "topic":           row['Topics'],
        "user_input":      row['Questions'],
        "reference":       str(row['Answers']).strip(),
        "model_response":  prediction,
        "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

responses_df = pd.DataFrame(records)
print(f"✅ Generated {len(responses_df)} responses")

Generating responses...


100%|██████████| 5/5 [03:03<00:00, 36.63s/it]

✅ Generated 5 responses


In [20]:
import torch, math

def calculate_perplexity_per_sample(model, tokenizer, topic, question, answer):
    """Returns perplexity for a single sample"""
    full_text = f"""[INST]
                    Topic: {topic}
                    User: {question}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    {answer}"""

    inputs    = tokenizer(full_text, return_tensors="pt",
                          truncation=True, max_length=2048).to("cuda")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs      = model(**inputs)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        loss         = torch.nn.CrossEntropyLoss(reduction="mean")(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

    ppl = math.exp(loss.item()) if math.isfinite(loss.item()) else float("inf")
    return round(ppl, 2)


print("Calculating perplexity per sample...")
perplexities = []

for row in tqdm(test_ds):
    # Perplexity on REFERENCE answer
    ppl_ref = calculate_perplexity_per_sample(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions'], row['Answers']
    )
    # Perplexity on MODEL'S OWN response
    model_resp = responses_df.loc[
        responses_df['user_input'] == row['Questions'], 'model_response'
    ].values
    ppl_model = calculate_perplexity_per_sample(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions'],
        model_resp[0] if len(model_resp) > 0 else row['Answers']
    )
    perplexities.append({
        "ppl_reference": ppl_ref,
        "ppl_model":     ppl_model
    })

ppl_df = pd.DataFrame(perplexities)
responses_df['ppl_reference'] = ppl_df['ppl_reference'].values
responses_df['ppl_model']     = ppl_df['ppl_model'].values

print(f"✅ Avg Perplexity (Reference): {responses_df['ppl_reference'].mean():.2f}")
print(f"✅ Avg Perplexity (Model):     {responses_df['ppl_model'].mean():.2f}")

Calculating perplexity per sample...


100%|██████████| 5/5 [00:05<00:00,  1.08s/it]

✅ Avg Perplexity (Reference): 1.73
✅ Avg Perplexity (Model):     1.98


In [21]:
from IPython.display import display, HTML

def show_comparison_table(df, num_rows=None):
    show_df = df.head(num_rows) if num_rows else df

    rows_html = ""
    for i, row in show_df.iterrows():
        rows_html += f"""
        <tr>
            <td style="background:#f0f4ff;font-weight:bold;padding:8px;vertical-align:top;min-width:120px">
                {row['topic']}
            </td>
            <td style="background:#fff8e1;padding:8px;vertical-align:top;min-width:200px">
                {row['user_input']}
            </td>
            <td style="background:#e8f5e9;padding:8px;vertical-align:top;min-width:200px">
                {row['reference']}
            </td>
            <td style="background:#fce4ec;padding:8px;vertical-align:top;min-width:200px">
                {row['model_response']}
            </td>
            <td style="text-align:center;padding:8px;vertical-align:top;background:#f3e5f5">
                {row['ppl_reference']}
            </td>
            <td style="text-align:center;padding:8px;vertical-align:top;background:#e0f7fa">
                {row['ppl_model']}
            </td>
        </tr>
        """

    html = f"""
    <style>
        table {{ border-collapse: collapse; width: 100%; font-size: 13px; font-family: Arial; }}
        th    {{ background: #37474f; color: white; padding: 10px; text-align: left; }}
        tr:hover td {{ filter: brightness(0.97); }}
    </style>
    <table>
        <thead>
            <tr>
                <th>📌 Topic</th>
                <th>💬 User Input</th>
                <th>📖 Reference Answer</th>
                <th>🤖 Model Response</th>
                <th>📊 PPL (Ref)</th>
                <th>📊 PPL (Model)</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    <br>
    <b>Summary:</b> 
    Avg PPL Reference = <b>{df['ppl_reference'].mean():.2f}</b> | 
    Avg PPL Model = <b>{df['ppl_model'].mean():.2f}</b> | 
    Total Samples = <b>{len(df)}</b>
    """
    display(HTML(html))

# Show all rows
show_comparison_table(responses_df)

📌 Topic,💬 User Input,📖 Reference Answer,🤖 Model Response,📊 PPL (Ref),📊 PPL (Model)
আশাবাদী,ভয় সাধারণত সব লাগে।,আমি শুনেছি ভয় মানুষকে সব ধরনের পাগলামি করতে বাধ্য করে।,"হ্যা, এটা ঠিক। অনেক মানুষ চিরকাল পরে জিতে থাকে! খোড়া ঘৃণা. আমার মনে হয় ফাইনালে উচ্ছেদের সঙ্গে অনেক মানুষ নেওয়ার জন্য দৌড়াবে - আপনি কি? :) (আমার ক্রস-গঞ্জ জন্য) আমার বন্ধুর কোকো নামে একটি নৈশ কুকুর আছে যে আমি সাধারণত ঝাঁকুনি পেয়েছি। কিন্তু আমার কখনো সৎ বাজি নেই। আমি সর্বদা এটা না পেতে. আমি এখনও আশা করি যে আমি এই বছর জিততে পারি.. LOL!! lol!!! আমি আশা করি তোমার দিন চমৎকার! এটা কখন আসে? প্রিয় অন্তরঙ্গ, আপনার পছন্দের পরিবেশটি সর",1.86,2.35
হতাশ,আপনি গত রাতে যে লটারির টিকিট কিনেছিলেন তা জেতেনি।,"হাহাহা, ভাল যে আমি অনুমান প্রত্যাশিত. আপনি কি অনেক লটারি খেলেন?","ওহ, এই ভীষণ খারাপ! অন্তত আমি বড় সঞ্চয়. ঠিক আছে, দেরি করে ঘুম থেকে উঠো। :) আপনি কখন পরে ফিরে আসবেন? এটা হতাশাজনক হতে পারে যদি আপনি ধনী হতে যাচ্ছেন. শুধু একটি ঝরঝরে দিন হবে! LOL!!! হাঃ হাঃ হাঃ! আমি এইমাত্র সঙ্গে আমার প্রথম স্ব-পরিবেশনের সৈকত বাড়িতে এসেছি এবং আমি বিশ্বাস করতে পারছি না যে আমি $৩০ এর বেশি স্ক্র্যাচ করতে পারি... এটি একটি বোনাস, আমি প্রায় প্রতি মাসে একটি জিতেছি। আমি এই বছর এই লটারির জন্য অপেক্ষা করতে পারি না কারণ আমি",1.77,1.85
একাকী,আমি গির্জা এবং বারে যাওয়ার চেষ্টা করেছি। আমি একা থাকতে খুব অপছন্দ করি.,বিভিন্ন ভিড়ের একটি ভাল মিশ্রণ মত মনে হচ্ছে,"হ্যালো, সিনেমা ভালো. শহরের উঠোনে ঘুরে ফিরতে ইঞ্জিন ধৌড় করতে পারেন! এটি প্রচুর ঝড়-বিনিময়ের মধ্যে আপনাকে প্রথমে যেতে সাহায্য করতে পারে। একটি বাড়ির পরিবর্তে একটি ঐতিহ্যগত ""কম্পিউন্ড"" কিছু করতে হবে? এটি একটি ভাল জায়গায় শুরু করার জন্য আপনাকে প্রথমে তাদের দক্ষতা দেখতে হবে (যদি আপনি একজন সঙ্গী না হন)!! :) এই পরিস্থিতিতে আপনি কোন বিকল্প করেছেন? একটি নতুন শহরে থাকা কঠিন হতে পারে..আপনি যদি স্থানীয় কমিউনিটিতে ঢুকতে পারেন তবে এ�",1.68,1.99
রাগান্বিত,আমি পড়াশুনা করি কিন্তু আমি স্মার্ট নই।,যতক্ষণ না তারা স্মার্ট হতে না শেখে ততক্ষণ পর্যন্ত কেউ স্মার্ট হয় না। আমরা সবাই মস্তিষ্ক নিয়ে জন্মেছি কিন্তু বুদ্ধি নিয়ে নয় আমাদের বুদ্ধি অর্জন করা উচিত।,"এখন, যে জন্য দোষ. হঠাৎ চিন্তা করলেন? অথবা ভাঙ্গা-ফেটে। আপনি শিখেছেন না তবে আপনি ঘৃণা করেন! উত্তেজনাপূর্ণ নয় :) আপনি আপনার প্রধান অবস্থান কী? আমি সত্যিই ঈর্ষান্বিত যে আপনি আপনার নতুন কাজের জন্য একটি নতুন ওয়্যার পেয়েছেন...কিন্তু আমি মনে করি এটি একটি ঝরঞ্জা (পার্টিতে) হতে পারে! পরের বার আপনার সাথে দেখা হবে! আমি আপনাকে আবার দেখতে পাব। শুধু কঠোর অধ্যয়ন! পৌরসভার কাউন্সিলের সদস্য হওয়ার জন্য আমাকে অনেক কঠোর অধ্যয়ন করতে হয়",1.74,1.91
বিস্মিত,হ্যাঁ আমি সত্যিই খুশি ছিল,আমি হব. এর আগে কেউ আমাকে গাড়ি কিনে দেয়নি।,"এটা চমৎকার! অনেক দিন পর গরীভ থাকা উচিত। :) ঈশ্বর, ওড়ার জন্য ঠাণ্ডা ঝঞ্জাট ঘৃষ্ণা. ১৫° F এর কম -৩০-৪০ °F? (আমি এটা করতে পারি না) আপনি যদি কোন পাঠ্য পড়তে পারেন এবং এটি দেখতে পারেন এবং আপনি এটি শুনতে পারেন তবে আপনার জন্য এটি একটি সঙ্গীতের রক্ষণাবেক্ষণ! আপনি কখনই ভাবতে পারবেন না... একটি গ্রীষ্ম সম্পর্কে! :D ) :) :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :( :(",1.59,1.79


In [28]:
# ── SQLite logging ─────────────────────────────────────────────────────────
conn = sqlite3.connect("experiments.db")
c    = conn.cursor()

c.execute("""CREATE TABLE IF NOT EXISTS LLAMAExperiments (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    model_name TEXT, lora_config TEXT,
    train_loss REAL, val_loss REAL,
    perplexity REAL, avg_bleu REAL,
    avg_rouge1 REAL, avg_rougel REAL,
    avg_human_empathy REAL, timestamp TEXT
)""")

c.execute("""CREATE TABLE IF NOT EXISTS GeneratedResponses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id INTEGER, input_text TEXT,
    response_text TEXT, timestamp TEXT
)""")

# ✅ Fix: compute ppl_score from responses_df (set in Cell 13)
ppl_score = float(responses_df['ppl_model'].mean())

c.execute("""INSERT INTO LLAMAExperiments 
    (model_name, lora_config, train_loss, val_loss, perplexity, 
     avg_bleu, avg_rouge1, avg_rougel, avg_human_empathy, timestamp)
    VALUES (?,?,?,?,?,?,?,?,?,?)""", (
    "LLaMA-3.1-8B-Instruct",
    "r16_alpha16_Unsloth",
    None,
    None,
    round(ppl_score, 2),
    round(float(metrics_df['bleu'].mean()), 4),
    round(float(metrics_df['rouge1'].mean()), 4),
    round(float(metrics_df['rougeL'].mean()), 4),
    round(float(avg_human_score), 2) if avg_human_score else None,
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
))
exp_id = c.lastrowid

for _, row in metrics_df.iterrows():
    c.execute("""INSERT INTO GeneratedResponses
        (experiment_id, input_text, response_text, timestamp)
        VALUES (?,?,?,?)""",
        (exp_id, row['input'], row['prediction'], row['timestamp']))

conn.commit()
conn.close()

# ── Also save to CSV ────────────────────────────────────────────────────────
metrics_df.to_csv("evaluation_results.csv", index=False)

summary = pd.DataFrame([{
    "model":            "LLaMA-3.1-8B-Instruct (LoRA/Unsloth)",
    "perplexity":       round(ppl_score, 2),
    "avg_bleu":         round(float(metrics_df['bleu'].mean()),   4),
    "avg_rouge1":       round(float(metrics_df['rouge1'].mean()), 4),
    "avg_rougeL":       round(float(metrics_df['rougeL'].mean()), 4),
    "avg_human_empathy": round(avg_human_score, 2) if avg_human_score else "N/A"
}])

print("\n" + "="*60)
print("         FINAL EVALUATION SUMMARY")
print("="*60)
print(summary.to_string(index=False))
print("\n✅ Saved: experiments.db  |  evaluation_results.csv")


         FINAL EVALUATION SUMMARY
                               model  perplexity  avg_bleu  avg_rouge1  avg_rougeL  avg_human_empathy
LLaMA-3.1-8B-Instruct (LoRA/Unsloth)        1.98     0.003         0.0         0.0                4.2

✅ Saved: experiments.db  |  evaluation_results.csv


In [29]:
pd.read_csv("evaluation_results.csv")

,topic,input,reference,prediction,bleu,rouge1,rougeL,timestamp
0,আশাবাদী,ভয় সাধারণত সব লাগে।,আমি শুনেছি ভয় মানুষকে সব ধরনের পাগলামি করতে ব...,"হ্যাঁ, আমি আশা করি আপনি অনুভব করেন যে আপনার কা...",0.002295,0.0,0,2026-03-12 10:35:09
1,হতাশ,আপনি গত রাতে যে লটারির টিকিট কিনেছিলেন তা জেতেনি।,"হাহাহা, ভাল যে আমি অনুমান প্রত্যাশিত. আপনি কি ...",আমি জানি এটা সত্যিই হতাশাজনক হতে পারে. আপনি কত...,0.005421,0.0,0,2026-03-12 10:35:45
2,একাকী,আমি গির্জা এবং বারে যাওয়ার চেষ্টা করেছি। আমি ...,বিভিন্ন ভিড়ের একটি ভাল মিশ্রণ মত মনে হচ্ছে,এটা কি সত্যি? আপনি কি এখনও নামাজ পড়েন? আমি আশ...,0.005107,0.0,0,2026-03-12 10:36:21
3,রাগান্বিত,আমি পড়াশুনা করি কিন্তু আমি স্মার্ট নই।,যতক্ষণ না তারা স্মার্ট হতে না শেখে ততক্ষণ পর্য...,"আমি নিশ্চিত আপনি কঠোর পড়াশোনা করছেন, কারণ আপন...",0.000000,0.0,0,2026-03-12 10:36:57
4,বিস্মিত,হ্যাঁ আমি সত্যিই খুশি ছিল,আমি হব. এর আগে কেউ আমাকে গাড়ি কিনে দেয়নি।,আমি খুশি যে আপনি একটি বিস্ময়কর জন্মদিন ছিল. আ...,0.002387,0.0,0,2026-03-12 10:37:33
